# Portable D FASTA smoke

This notebook is a step-by-step wrapper around the local release candidate. It does not download weights or publish a model. Before running inference, stage the verified seed-42 arm-D `pytorch_model.bin` at `../bundle/model/pytorch_model.bin`.

In [ ]:
from pathlib import Path
import json
root = Path('..').resolve()
bundle = root / 'bundle'
print(json.dumps(json.loads((bundle/'manifest.json').read_text()), indent=2))
print('weights present:', (bundle/'model'/'pytorch_model.bin').is_file())

## Install

From a shell in `release_candidate/`, install the target torch build and then `python -m pip install -e .`. The package does not choose a CUDA wheel for you.

In [ ]:
# Run this cell only after installing the package and staging the checkpoint.
# !python -m pip install -e ..

## Run CPU or GPU inference

The same command works on CPU and CUDA. Use a fresh output directory for each run.
The bundled human hs1 example uses two already observed 4096-bp DEV windows. The fixed CPU run produced 3,418 positive bases; this is a reproducible output example, not an independent accuracy test.

In [ ]:
import subprocess, sys
fasta = root / 'examples' / 'human_hs1_example.fa'
out = root / 'notebook-smoke-output'
command = [sys.executable, '-m', 'portable_d', '--bundle-root', str(bundle), '--fasta', str(fasta), '--output-dir', str(out), '--device', 'cpu', '--batch-size', '2']
print(' '.join(command))
# subprocess.run(command, check=True)  # uncomment after weights are staged

## Inspect products

A completed run writes probability bedGraph, threshold-positive material BED, an ambiguity QC BED, softmasked FASTA, and a final summary. These are material tracks and do not encode insertion identities or family labels.

In [ ]:
if (out/'summary.json').is_file():
    summary = json.loads((out/'summary.json').read_text())
    print(summary['status'], summary['model_id'], summary['device'])
    for key, path in summary['outputs'].items():
        print(key, path)
else:
    print('No completed run yet')